# MeTRAbs Pose Estimation Test
Test your iPhone ARKit video + camera intrinsics JSON with MeTRAbs.

**Runtime > Change runtime type > GPU (T4)** before running!

In [ ]:
# Step 1: Install dependencies
!pip install -q tensorflow opencv-python-headless certifi

In [ ]:
# Step 2: Verify GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {gpus}')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU ready!')
else:
    print('WARNING: No GPU - go to Runtime > Change runtime type > GPU')

In [ ]:
# Step 3: Upload your video and JSON
# Click the upload button and select both files from your iPhone
from google.colab import files
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

In [ ]:
# Step 4: Parse your camera intrinsics JSON
import json
import numpy as np
import glob

# Find the JSON file
json_file = [f for f in uploaded.keys() if f.endswith('.json')][0]
video_file = [f for f in uploaded.keys() if f.endswith(('.mp4', '.mov'))][0]

with open(json_file) as f:
    camera_data = json.load(f)

print('=== Camera Data ===')
print(f"Resolution: {camera_data['image_resolution']}")
print(f"FPS: {camera_data['recording_info']['target_fps']}")
print(f"Frames: {camera_data['recording_info']['frame_count']}")
print(f"LiDAR: {camera_data['depth_data']['has_lidar']}")
print(f"Depth: {camera_data['depth_data']['has_depth']}")

# Extract the 3x3 intrinsic matrix
intrinsic_matrix = np.array(
    camera_data['camera_intrinsics']['intrinsic_matrix'],
    dtype=np.float32
)
print(f'\n=== Intrinsic Matrix (3x3) ===')
print(intrinsic_matrix)
print(f'\nfx={intrinsic_matrix[0,0]:.2f}, fy={intrinsic_matrix[1,1]:.2f}')
print(f'cx={intrinsic_matrix[0,2]:.2f}, cy={intrinsic_matrix[1,2]:.2f}')

In [ ]:
# Step 5: Download and load MeTRAbs model
import os, ssl, certifi

ssl_context = ssl.create_default_context(cafile=certifi.where())
ssl._create_default_https_context = lambda: ssl_context

MODEL_TYPE = 'metrabs_mob3l_y4t'
CACHE_DIR = './metrabs_models'
os.makedirs(CACHE_DIR, exist_ok=True)

fname = f'{MODEL_TYPE}_20211019.zip'
origin = f'https://omnomnom.vision.rwth-aachen.de/data/metrabs/{fname}'

print('Downloading MeTRAbs model (first time only)...')
model_zip = tf.keras.utils.get_file(
    fname=fname, origin=origin, extract=True,
    cache_subdir='.', cache_dir=CACHE_DIR
)

# Find saved_model.pb
model_path = None
for root, dirs, fnames in os.walk(os.path.dirname(model_zip)):
    if 'saved_model.pb' in fnames:
        model_path = root
        break

print(f'Loading model from: {model_path}')
model = tf.saved_model.load(model_path)
print('Model loaded!')

In [ ]:
# Step 6: Run MeTRAbs on your video WITH intrinsics
import cv2
from time import time
from IPython.display import HTML
from base64 import b64encode

SKELETON = 'smpl_24'
RESIZE = 0.5  # process at half resolution for speed

cap = cv2.VideoCapture(video_file)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out_w, out_h = int(width * RESIZE), int(height * RESIZE)

print(f'Video: {width}x{height} @ {fps} FPS')
print(f'Processing at: {out_w}x{out_h}')

# Scale intrinsics to match resized frames
scaled_intrinsics = intrinsic_matrix.copy()
scaled_intrinsics[0, :] *= RESIZE  # scale fx, cx
scaled_intrinsics[1, :] *= RESIZE  # scale fy, cy
intrinsics_tensor = tf.constant(scaled_intrinsics, dtype=tf.float32)
print(f'Scaled intrinsics for {out_w}x{out_h}:')
print(scaled_intrinsics)

# Get skeleton edges for drawing
edges = model.per_skeleton_joint_edges[SKELETON].numpy().tolist()

# Output video
output_path = 'output_poses.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (out_w, out_h))

# Store all 3D poses
all_poses_3d = []
frame_idx = 0
t0 = time()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    frame_small = cv2.resize(frame, (out_w, out_h))
    rgb = cv2.cvtColor(frame_small, cv2.COLOR_BGR2RGB)

    with tf.device('/GPU:0'):
        image_tensor = tf.convert_to_tensor(rgb)
        # THIS is the key line - passing YOUR camera intrinsics!
        pred = model.detect_poses(
            image_tensor,
            skeleton=SKELETON,
            intrinsic_matrix=intrinsics_tensor
        )

    # Extract results
    poses2d = pred.get('poses2d', None)
    poses3d = pred.get('poses3d', None)

    if hasattr(poses2d, 'numpy'):
        poses2d = poses2d.numpy()
    if hasattr(poses3d, 'numpy'):
        poses3d_np = poses3d.numpy()
        all_poses_3d.append({
            'frame': frame_idx,
            'poses3d': poses3d_np.tolist()
        })

    # Draw skeletons
    if poses2d is not None:
        for pose in poses2d:
            pts = pose.astype(int)
            for i, j in edges:
                if i < len(pts) and j < len(pts):
                    cv2.line(frame_small, tuple(pts[i]), tuple(pts[j]), (0, 200, 255), 2)
            for x, y in pts:
                if 0 <= x < out_w and 0 <= y < out_h:
                    cv2.circle(frame_small, (int(x), int(y)), 3, (0, 128, 255), -1)

    out.write(frame_small)

    if frame_idx % 30 == 0:
        elapsed = time() - t0
        print(f'Frame {frame_idx} | {elapsed:.1f}s | {frame_idx/elapsed:.1f} FPS')

cap.release()
out.release()

elapsed = time() - t0
print(f'\nDone! {frame_idx} frames in {elapsed:.1f}s ({frame_idx/elapsed:.1f} FPS)')
print(f'Output saved to: {output_path}')

In [ ]:
# Step 7: Save 3D poses as CSV
import csv

csv_path = 'poses_3d.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['frame', 'person_id', 'joint_id', 'x_mm', 'y_mm', 'z_mm'])
    for entry in all_poses_3d:
        for person_id, pose in enumerate(entry['poses3d']):
            for joint_id, (x, y, z) in enumerate(pose):
                writer.writerow([entry['frame'], person_id, joint_id, x, y, z])

print(f'3D poses saved to {csv_path}')
print(f'Total frames with detections: {len(all_poses_3d)}')

In [ ]:
# Step 8: Play the output video in Colab
# First compress with ffmpeg for browser playback
!ffmpeg -i output_poses.mp4 -vcodec libx264 -crf 28 -y output_poses_h264.mp4 2>/dev/null

from IPython.display import HTML
from base64 import b64encode

video_data = open('output_poses_h264.mp4', 'rb').read()
b64 = b64encode(video_data).decode()
HTML(f'''
<video width="600" controls>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
''')

In [ ]:
# Step 9: Download results
from google.colab import files
files.download('output_poses_h264.mp4')
files.download('poses_3d.csv')

In [ ]:
# BONUS: Compare WITH vs WITHOUT intrinsics
# Run the same frame without intrinsics to see the difference

cap = cv2.VideoCapture(video_file)
ret, frame = cap.read()
cap.release()

if ret:
    frame_small = cv2.resize(frame, (out_w, out_h))
    rgb = cv2.cvtColor(frame_small, cv2.COLOR_BGR2RGB)
    image_tensor = tf.convert_to_tensor(rgb)

    # WITH intrinsics
    pred_with = model.detect_poses(
        image_tensor, skeleton=SKELETON,
        intrinsic_matrix=intrinsics_tensor
    )

    # WITHOUT intrinsics
    pred_without = model.detect_poses(
        image_tensor, skeleton=SKELETON
    )

    if 'poses3d' in pred_with and 'poses3d' in pred_without:
        p3d_with = pred_with['poses3d'].numpy()
        p3d_without = pred_without['poses3d'].numpy()

        if len(p3d_with) > 0 and len(p3d_without) > 0:
            diff = np.abs(p3d_with[0] - p3d_without[0])
            print('=== 3D Pose Difference (WITH vs WITHOUT intrinsics) ===')
            print(f'Mean difference per joint: {diff.mean(axis=1).round(1)} mm')
            print(f'Overall mean difference: {diff.mean():.1f} mm')
            print(f'Max difference: {diff.max():.1f} mm')
            print(f'\nThis shows how much MORE ACCURATE your poses are')
            print(f'when using the real iPhone camera intrinsics!')
        else:
            print('No person detected in first frame')
    else:
        print('No 3D poses returned')